In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from transformers import DistilBertTokenizerFast, DistilBertModel
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler
import numpy as np
from collections import Counter
import random
from tqdm import tqdm
from sklearn.metrics import f1_score

MAX_LEN = 512
BATCH_SIZE = 64
EPOCHS = 3
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load data
df = pd.read_csv("/content/drive/MyDrive/datasets/kaggle_test.csv")
texts = df["text"].astype(str).tolist()
labels = df["generated"].astype(int).tolist()

In [ ]:
from datasets import load_dataset

ds = load_dataset("artem9k/ai-text-detection-pile")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00007-bc5952582e004d(…):   0%|          | 0.00/758M [00:00<?, ?B/s]

data/train-00001-of-00007-71c80017bc45f3(…):   0%|          | 0.00/318M [00:00<?, ?B/s]

data/train-00002-of-00007-ee2d43f396e78f(…):   0%|          | 0.00/125M [00:00<?, ?B/s]

data/train-00003-of-00007-529931154b42b5(…):   0%|          | 0.00/137M [00:00<?, ?B/s]

data/train-00004-of-00007-b269dc49374a2c(…):   0%|          | 0.00/137M [00:00<?, ?B/s]

data/train-00005-of-00007-3dce5e05ddbad7(…):   0%|          | 0.00/258M [00:00<?, ?B/s]

data/train-00006-of-00007-3d8a471ba0cf1c(…):   0%|          | 0.00/242M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1392522 [00:00<?, ? examples/s]

In [ ]:
df_pile = pd.DataFrame(ds['train'])[['text', 'source']]

df_pile = df_pile.rename(columns={'source': 'label'})
df_pile['label'] = df_pile['label'].map({'human': 0, 'ai': 1})

In [ ]:
df = df.rename(columns={'generated': 'label'})

In [ ]:
df_pile['label'].value_counts()

,count
label,
0,1028146
1,364376


In [ ]:
# Combine Kaggle + Pile
df_combined = pd.concat([df, df_pile], ignore_index=True)
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

print("Combined dataset size:", len(df_combined))
print(df_combined['label'].value_counts())

Combined dataset size: 1879757
label
0.0    1333943
1.0     545814
Name: count, dtype: int64


In [ ]:
# Separate features and labels
X_all = df_combined['text']
y_all = df_combined['label']

In [ ]:
# Train/Val Split (+ small test size 10%)
X_train_val, X_test_internal, y_train_val, y_test_internal = train_test_split(
    X_all, y_all, test_size=0.1, random_state=42, stratify=y_all
)

# 80/20 train/val
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.2, random_state=42, stratify=y_train_val
)

X_train_text = X_train.tolist()
X_val_text = X_val.tolist()
X_test_text_internal = X_test_internal.tolist()

y_train = y_train.tolist()
y_val = y_val.tolist()
y_test_internal = y_test_internal.tolist()

print("Train:", len(X_train_text), "Val:", len(X_val_text), "Internal Test:", len(X_test_text_internal))
print("Class distribution in train:", Counter(y_train))
print("Class distribution in val:", Counter(y_val))
print("Class distribution in internal test:", Counter(y_test_internal))

Train: 1353424 Val: 338357 Internal Test: 187976
Class distribution in train: Counter({0.0: 960438, 1.0: 392986})
Class distribution in val: Counter({0.0: 240110, 1.0: 98247})
Class distribution in internal test: Counter({0.0: 133395, 1.0: 54581})


In [ ]:
# Tokenization
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def chunk_texts(texts, labels, tokenizer, max_len=MAX_LEN, batch_size=512):
    all_chunks = []
    all_chunk_labels = []

    texts = [str(t) for t in texts]

    for i in tqdm(range(0, len(texts), batch_size), desc="Batch tokenizing"):
        batch_texts = texts[i:i+batch_size]
        batch_labels = labels[i:i+batch_size]

        batch_encodings = tokenizer(batch_texts, add_special_tokens=True, return_attention_mask=False, truncation=False)

        for input_ids, label in zip(batch_encodings['input_ids'], batch_labels):
            for j in range(0, len(input_ids), max_len):
                chunk = input_ids[j:j+max_len]
                all_chunks.append(chunk)
                all_chunk_labels.append(label)

    return all_chunks, all_chunk_labels

X_train_chunks, y_train_chunks = chunk_texts(X_train, y_train, tokenizer)
X_val_chunks, y_val_chunks = chunk_texts(X_val, y_val, tokenizer)
X_test_chunks, y_test_chunks = chunk_texts(X_test_internal, y_test_internal, tokenizer)

print("Train chunks:", len(X_train_chunks))
print("Val chunks:", len(X_val_chunks))
print("Test chunks:", len(X_test_chunks))


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Batch tokenizing: 100%|██████████| 368/368 [01:01<00:00,  5.96it/s]

Train chunks: 2008289
Val chunks: 501623
Test chunks: 279767


In [ ]:
class BertDataset(Dataset):
    def __init__(self, encodings, labels):
        self.input_ids_list = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.input_ids_list[idx], dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

def collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]
    labels = torch.stack([item["labels"] for item in batch])
    input_ids_padded = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = (input_ids_padded != tokenizer.pad_token_id).long()
    return {"input_ids": input_ids_padded, "attention_mask": attention_mask, "labels": labels}

train_dataset = BertDataset(X_train_chunks, y_train_chunks)
val_dataset   = BertDataset(X_val_chunks, y_val_chunks)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn)

In [ ]:
class DistilBERTClassifier(nn.Module):
    def __init__(self, dropout=0.3):
        super(DistilBERTClassifier, self).__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:,0]  # CLS token
        x = self.dropout(pooled_output)
        x = self.classifier(x)
        return x

model = DistilBERTClassifier().to(DEVICE)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [ ]:
num_ai = sum(y_train_chunks)
num_human = len(y_train_chunks) - num_ai
pos_weight = torch.tensor(num_human / num_ai).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = AdamW(model.parameters(), lr=2e-5)

In [ ]:
scaler = GradScaler()
best_val_loss = float("inf")
best_threshold = 0.5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].unsqueeze(1).float().to(DEVICE)

        with autocast():
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].unsqueeze(1).float().to(DEVICE)

            with autocast():
                outputs = model(input_ids, attention_mask)
                loss = criterion(outputs, labels)

            val_loss += loss.item()
            all_logits.extend(outputs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    avg_val_loss = val_loss / len(val_loader)

    probs = torch.sigmoid(torch.tensor(all_logits)).numpy().flatten()
    true = np.array(all_labels).flatten()

    thresholds = np.linspace(0.05, 0.95, 50)
    best_f1 = -1
    best_t = 0.5

    for t in thresholds:
        preds = (probs >= t).astype(int)
        f1 = f1_score(true, preds)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    best_threshold = best_t
    val_preds = (probs >= best_threshold).astype(int)
    val_acc = accuracy_score(true, val_preds)

    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f} | "
          f"Best Threshold: {best_threshold:.3f} | F1: {best_f1:.4f}")

    # Save checkpoint
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({
            "model_state_dict": model.state_dict(),
            "threshold": best_threshold
        }, "best_distilbert_model.pt")


/tmp/ipython-input-1778042840.py:1: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Epoch 1:   0%|          | 0/31380 [00:00<?, ?it/s]/tmp/ipython-input-1778042840.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 1: 100%|██████████| 31380/31380 [55:48<00:00,  9.37it/s]
/tmp/ipython-input-1778042840.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipython-input-1778042840.py:47: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  probs = torch.sigmoid(torch.tensor(all_

Epoch 1 | Train Loss: 0.1025 | Val Loss: 0.1235 | Val Acc: 0.9709 | Best Threshold: 0.858 | F1: 0.9475


Epoch 2:   0%|          | 0/31380 [00:00<?, ?it/s]/tmp/ipython-input-1778042840.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 2: 100%|██████████| 31380/31380 [55:32<00:00,  9.42it/s]
/tmp/ipython-input-1778042840.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 2 | Train Loss: 0.0552 | Val Loss: 0.0729 | Val Acc: 0.9808 | Best Threshold: 0.821 | F1: 0.9651


Epoch 3:   0%|          | 0/31380 [00:00<?, ?it/s]/tmp/ipython-input-1778042840.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 3: 100%|██████████| 31380/31380 [55:29<00:00,  9.42it/s]
/tmp/ipython-input-1778042840.py:37: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 3 | Train Loss: 0.0384 | Val Loss: 0.0776 | Val Acc: 0.9801 | Best Threshold: 0.748 | F1: 0.9639


In [ ]:
test_dataset   = BertDataset(X_test_chunks, y_test_chunks)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

In [ ]:
model = DistilBERTClassifier().to(DEVICE)
checkpoint = torch.load("best_distilbert_model.pt", map_location=DEVICE, weights_only=False)

state_dict = checkpoint["model_state_dict"]

model.load_state_dict(state_dict)
model.eval()

DistilBERTClassifier(
  (bert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): L

In [ ]:
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].float().to(DEVICE)

        logits = model(input_ids, attention_mask)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).long()

        all_preds.extend(preds.cpu().numpy().flatten())
        all_labels.extend(labels.cpu().numpy().flatten())

# Metrics
test_acc = accuracy_score(all_labels, all_preds)
test_report = classification_report(all_labels, all_preds, digits=4)

print("Test Accuracy:", test_acc)
print("Classification Report:\n", test_report)

Evaluating: 100%|██████████| 4372/4372 [13:11<00:00,  5.52it/s]


Test Accuracy: 0.9751972176847162
Classification Report:
               precision    recall  f1-score   support

         0.0     0.9975    0.9683    0.9827    203115
         1.0     0.9220    0.9935    0.9564     76652

    accuracy                         0.9752    279767
   macro avg     0.9598    0.9809    0.9695    279767
weighted avg     0.9768    0.9752    0.9755    279767



In [ ]:
df_test = pd.read_csv("/content/drive/MyDrive/datasets/ieee_train.csv")
X_test2_text = df_test["text"].astype(str).tolist()
y_test2 = df_test["label"].astype(int).tolist()

In [ ]:
df_test["label"].value_counts()

,count
label,
0,5000
1,5000


In [ ]:
def chunk_texts2(texts, labels, tokenizer, max_length=512):
    X_chunks = []
    y_chunks = []
    chunk_to_article_idx = []

    for idx, (text, label) in enumerate(zip(texts, labels)):
        encodings = tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=max_length,
            return_tensors='pt'
        )

        X_chunks.append(encodings['input_ids'].squeeze(0))
        y_chunks.append(label)
        chunk_to_article_idx.append(idx)

    return X_chunks, y_chunks, chunk_to_article_idx

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

X_test2_chunks, y_test2_chunks, chunk_to_article_idx = chunk_texts2(X_test2_text, y_test2, tokenizer)

In [ ]:
test2_dataset = BertDataset(X_test2_chunks, y_test2_chunks)
test2_loader = DataLoader(test2_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)


In [ ]:
model = DistilBERTClassifier().to(DEVICE)
checkpoint = torch.load("best_distilbert_model.pt", map_location=DEVICE, weights_only=False)
state_dict = checkpoint["model_state_dict"]
model.load_state_dict(state_dict)
model.eval()

DistilBERTClassifier(
  (bert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): L

In [ ]:
all_probs = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test2_loader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].float().to(DEVICE)

        logits = model(input_ids, attention_mask)
        probs = torch.sigmoid(logits)

        all_probs.extend(probs.cpu().numpy().flatten())
        all_labels.extend(labels.cpu().numpy().flatten())

chunk_preds = [int(p >= 0.5) for p in all_probs]

chunk_acc = accuracy_score(all_labels, chunk_preds)
chunk_report = classification_report(all_labels, chunk_preds, digits=4)
print("Chunk-level Accuracy:", chunk_acc)
print("Chunk-level Classification Report:\n", chunk_report)

Evaluating:   0%|          | 0/157 [00:00<?, ?it/s]/tmp/ipython-input-966092127.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(self.input_ids_list[idx], dtype=torch.long),
Evaluating: 100%|██████████| 157/157 [00:27<00:00,  5.63it/s]

Chunk-level Accuracy: 0.8977
Chunk-level Classification Report:
               precision    recall  f1-score   support

         0.0     0.8866    0.9120    0.8991      5000
         1.0     0.9094    0.8834    0.8962      5000

    accuracy                         0.8977     10000
   macro avg     0.8980    0.8977    0.8977     10000
weighted avg     0.8980    0.8977    0.8977     10000



In [ ]:
from collections import defaultdict

In [ ]:
article_preds = defaultdict(list)
article_labels = {}

for i, article_idx in enumerate(chunk_to_article_idx):
    article_preds[article_idx].append(all_probs[i])  # keep probabilities
    article_labels[article_idx] = all_labels[i]      # same label for all chunks

final_preds = []
final_labels = []

for idx in article_preds:
    avg_prob = np.mean(article_preds[idx])
    label = int(avg_prob >= 0.5)
    final_preds.append(label)
    final_labels.append(article_labels[idx])

article_acc = accuracy_score(final_labels, final_preds)
article_report = classification_report(final_labels, final_preds, digits=4)

print("Article-level Accuracy:", article_acc)
print("Article-level Classification Report:\n", article_report)

Article-level Accuracy: 0.8977
Article-level Classification Report:
               precision    recall  f1-score   support

         0.0     0.8866    0.9120    0.8991      5000
         1.0     0.9094    0.8834    0.8962      5000

    accuracy                         0.8977     10000
   macro avg     0.8980    0.8977    0.8977     10000
weighted avg     0.8980    0.8977    0.8977     10000

